# Analysis of the Conservation State of Ancient Books

**Information Systems and Business Intelligence** — A.Y. 2025/2026

Run the setup cell first, then execute all cells in order.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def _run(cmd):
    subprocess.run(cmd, shell=True, check=False)

# Colab / Jupyter: clone repo if src/ is missing
if not Path('src').exists():
    _run('git clone https://github.com/kimias21/book-conservation-bi.git')
    os.chdir('book-conservation-bi')

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

_run('pip install -q -r requirements.txt')
if not Path('data/processed/integrated_heritage_books.csv').exists():
    _run(f'{sys.executable} -m src.data_integration')

import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.config import INTEGRATED_CSV, DATA_PROCESSED
from src.conservation_index import add_conservation_columns

sns.set_theme(style='whitegrid')
print('Project root:', PROJECT_ROOT)
print('Data file exists:', INTEGRATED_CSV.exists())


## 1. Loading and pre-processing

In [ ]:
df = pd.read_csv(INTEGRATED_CSV)
print(df.shape)
df.head()

In [ ]:
TAXONOMY = {'parchment':'Parchment','laid_paper':'Laid paper','industrial_paper':'Industrial paper','mixed':'Mixed support'}
df['support_material_label'] = df['support_material'].map(TAXONOMY)
for c in ['flood_event','hvac_failure_event','restoration_recent']:
    df[c] = df[c].astype(bool)
print('Missing:', df.isnull().sum().sum())
df.describe().T

Exploratory analysis of CRI by material, century, and environmental variables.


## 2. Exploratory analysis

In [ ]:
df['conservation_risk_index'].hist(bins=30)
plt.title('CRI distribution'); plt.show()
sns.boxplot(data=df, x='support_material_label', y='conservation_risk_index')
plt.xticks(rotation=30); plt.show()
sns.heatmap(df[['conservation_risk_index','avg_relative_humidity_pct','avg_temperature_c','age_stress','material_stress']].corr(), annot=True)
plt.show()

## 3. Conservation Risk Index

In [ ]:
df = add_conservation_columns(df)
print(df['risk_level'].value_counts())
print(df[['conservation_risk_index','observed_condition_score']].corr())

## 4. Models

In [ ]:
groups = [g['conservation_risk_index'].values for _, g in df.groupby('support_material')]
print('ANOVA F,p:', stats.f_oneway(*groups))

# Target: elevated risk (CRI >= 45) — more stable than high_risk with few positives
df['elevated_risk'] = (df['conservation_risk_index'] >= 45).astype(int)
print('Elevated risk count:', df['elevated_risk'].sum())

feature_cols = ['century','avg_temperature_c','avg_relative_humidity_pct','avg_light_lux',
                'air_quality_index','support_material','ink_type','binding_type','site_type']
X, y = df[feature_cols], df['elevated_risk']
cat = ['support_material','ink_type','binding_type','site_type']
num = [c for c in feature_cols if c not in cat]
prep = ColumnTransformer([('num', StandardScaler(), num),
                          ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
split_args = dict(test_size=0.25, random_state=42)
if y.sum() >= 10 and (len(y) - y.sum()) >= 10:
    split_args['stratify'] = y
X_train, X_test, y_train, y_test = train_test_split(X, y, **split_args)
rf = Pipeline([('prep', prep),
               ('clf', RandomForestClassifier(200, random_state=42, class_weight='balanced'))])
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), zero_division=0))


### Chi-square test — flood exposure vs high risk

Testing whether volumes that experienced a flood event are statistically more likely to be classified as high risk.

In [ ]:
from scipy.stats import chi2_contingency

ct = pd.crosstab(df['high_risk'], df['flood_event'])
print('Contingency table:')
print(ct)
chi2, p, dof, expected = chi2_contingency(ct)
print(f'\nChi-square: χ²={chi2:.3f}, p={p:.4f}, dof={dof}')
if p < 0.05:
    print('Result: statistically significant association between flood events and high risk (p < 0.05)')
else:
    print('Result: no statistically significant association detected at α=0.05')

### Feature importance — random forest

Which variables drive the elevated-risk classification most? The chart shows the top 10 features by mean decrease in impurity.

In [ ]:
cat_features = list(
    rf.named_steps['prep']
    .named_transformers_['cat']
    .get_feature_names_out(cat)
)
all_features = num + cat_features
importances = rf.named_steps['clf'].feature_importances_
feat_series = pd.Series(importances, index=all_features).nlargest(10)

fig, ax = plt.subplots(figsize=(8, 4))
feat_series.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Mean decrease in impurity')
ax.set_title('Top 10 feature importances (random forest)')
plt.tight_layout()
plt.show()

### Geographic distribution of conservation risk

Each site is plotted by its coordinates. Bubble size reflects the number of volumes; colour reflects mean CRI. Sites in warm climates with higher humidity (Bologna, Florence) tend to show elevated risk.

In [ ]:
site_agg = (
    df.groupby(['site_id', 'site_name', 'latitude', 'longitude'], as_index=False)
    .agg(mean_cri=('conservation_risk_index', 'mean'),
         volume_count=('volume_id', 'count'))
)

fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    site_agg['longitude'],
    site_agg['latitude'],
    s=site_agg['volume_count'] * 3,
    c=site_agg['mean_cri'],
    cmap='YlOrRd',
    edgecolors='grey',
    linewidths=0.5,
    alpha=0.85
)
for _, row in site_agg.iterrows():
    ax.annotate(row['site_name'].split(',')[0],
                (row['longitude'], row['latitude']),
                fontsize=7, ha='left', va='bottom',
                xytext=(4, 4), textcoords='offset points')
plt.colorbar(scatter, ax=ax, label='Mean CRI')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Geographic distribution of mean conservation risk by site')
plt.tight_layout()
plt.show()

In [ ]:
ts = pd.read_csv(DATA_PROCESSED / 'environmental_timeseries.csv')
fig, ax = plt.subplots(figsize=(10,4))
for sid, g in ts.groupby("site_id"):
    ax.plot(g["month"], g["avg_relative_humidity_pct"], label=sid)
ax.set_ylabel("RH %"); ax.set_title("Temporal evolution of relative humidity")
ax.legend(ncol=4, fontsize=7); plt.xticks(rotation=45); plt.tight_layout(); plt.show()


### Clustering risk profiles

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
X = df[["environmental_stress","material_stress","age_stress","event_stress"]]
km = KMeans(4, random_state=42, n_init=10).fit(StandardScaler().fit_transform(X))
df["cluster"] = km.labels_
print(df.groupby("cluster")["conservation_risk_index"].mean())


## 5. Discussion

The clustering step shows that risk profiles separate mainly along century and site type. Medieval parchment volumes in Mediterranean archives consistently fall into the high-risk cluster, while 19th-century industrial paper volumes in northern European libraries land at low risk. Random forest feature importances confirm that age and average humidity are the dominant drivers — ink type contributed less than expected. The CRI weights are a reasonable starting point but would benefit from calibration against real conservator assessments.